In [ ]:
from pathlib import Path
import shutil
import subprocess

PROJECT_DIR = Path("/kaggle/working/Real-ESRGAN")
# Upload the delivered ZIP as a Kaggle dataset and update this path if needed.
PROJECT_ARCHIVE = Path("/kaggle/input/realesrgan-basicvsrpp/realesrgan_basicvsrpp_overlay.zip")
UPSTREAM_TAG = "v0.3.0"

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", UPSTREAM_TAG,
    "https://github.com/xinntao/Real-ESRGAN.git", str(PROJECT_DIR),
], check=True)
if not PROJECT_ARCHIVE.is_file():
    raise FileNotFoundError(
        f"Overlay archive not found: {PROJECT_ARCHIVE}. Upload the delivered ZIP and update PROJECT_ARCHIVE."
    )
subprocess.run(["unzip", "-q", "-o", str(PROJECT_ARCHIVE), "-d", str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], check=True)


In [ ]:
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile realesrgan.py enhance/*.py
!cd /kaggle/working/Real-ESRGAN && pytest -q


In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime2/cm_4.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan_basicvsrpp.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

# BasicVSR++ compressed-video enhancement runs before Real-ESRGAN.
BASICVSRPP = True
BASICVSRPP_TRACK = 1             # 1=fidelity; 2=perceptual; 3=fixed-bitrate fidelity
BASICVSRPP_MODEL_PATH = ""       # empty: download the official checkpoint
BASICVSRPP_GPU = 0
BASICVSRPP_FP16 = True
BASICVSRPP_CLIP_LENGTH = 7
BASICVSRPP_CLIP_OVERLAP = 2
BASICVSRPP_TILE_SIZE = 512        # lower to 256 if BasicVSR++ runs out of memory
BASICVSRPP_TILE_PAD = 32
BASICVSRPP_STRENGTH = 1.0
BASICVSRPP_SCENE_THRESHOLD = 0.30

NATIVE_ANALYSIS = "off"          # off / report / auto
NATIVE_SAMPLES = 5
NATIVE_MIN_HEIGHT = 500
NATIVE_MAX_HEIGHT = 1080
NATIVE_KERNELS = "bilinear,bicubic,lanczos"
NATIVE_CONFIDENCE = 0.85
NATIVE_HEIGHT = 0
NATIVE_KERNEL = "auto"
DESCALE = False

INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 256                   # Real-ESRGAN tile size; 0 uses whole frames
TILE_PAD = 10
PRE_PAD = 0
TILE_VERIFY_COVERAGE = True
BATCH_SIZE = 4
GPU_IDS = "0,1"

# Baseline Real-ESRGAN path. Enable extras only when deliberately testing them.
TTA = "none"
TTA_BATCH_SIZE = 1
SHIFT_ENSEMBLE = "none"
RESIDUAL_MODE = "official"
RESIDUAL_STRENGTH = 1.0
RESIDUAL_FLAT_STRENGTH = 0.9
RESIDUAL_EDGE_STRENGTH = 1.0
RESIDUAL_EDGE_LOW = 0.05
RESIDUAL_EDGE_HIGH = 0.20
BASE_CORRECTION = 0.0
BACK_PROJECTION_ITERATIONS = 0
BACK_PROJECTION_STRENGTH = 0.2
BACK_PROJECTION_KERNEL = "lanczos"
BACK_PROJECTION_CLAMP = 0.05
DEHALO_STRENGTH = 0.0
DEHALO_RADIUS = 2
RANGE_LIMIT = 0.0
RANGE_RADIUS = 2
OVERSHOOT = 1.0
UNDERSHOOT = 1.0

COLOR_POLICY = "preserve"        # preserve / bt709
HDR_POLICY = "reject"            # reject / passthrough

ANIME4K = False
ANIME4K_SHADER_DIR = ""
ANIME4K_SHADERS = ""
ANIME4K_STRENGTH = 1.0

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"

EXTRA_ARGS = []


In [ ]:
import shlex
import subprocess
import sys

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan.py",
    "--input", INPUT_VIDEO, "--output", OUTPUT_VIDEO,
    "--model", MODEL, "--model-path", MODEL_PATH,
    "--scale", str(SCALE), "--fps", FPS,
    "--fp16", "--channels-last",
    "--basicvsrpp" if BASICVSRPP else "--no-basicvsrpp",
    "--basicvsrpp-track", str(BASICVSRPP_TRACK),
    "--basicvsrpp-model-path", BASICVSRPP_MODEL_PATH,
    "--basicvsrpp-gpu", str(BASICVSRPP_GPU),
    "--basicvsrpp-fp16" if BASICVSRPP_FP16 else "--no-basicvsrpp-fp16",
    "--basicvsrpp-clip-length", str(BASICVSRPP_CLIP_LENGTH),
    "--basicvsrpp-clip-overlap", str(BASICVSRPP_CLIP_OVERLAP),
    "--basicvsrpp-tile-size", str(BASICVSRPP_TILE_SIZE),
    "--basicvsrpp-tile-pad", str(BASICVSRPP_TILE_PAD),
    "--basicvsrpp-strength", str(BASICVSRPP_STRENGTH),
    "--basicvsrpp-scene-threshold", str(BASICVSRPP_SCENE_THRESHOLD),
    "--native-analysis", NATIVE_ANALYSIS, "--native-samples", str(NATIVE_SAMPLES),
    "--native-min-height", str(NATIVE_MIN_HEIGHT), "--native-max-height", str(NATIVE_MAX_HEIGHT),
    "--native-kernels", NATIVE_KERNELS, "--native-confidence", str(NATIVE_CONFIDENCE),
    "--native-height", str(NATIVE_HEIGHT), "--native-kernel", NATIVE_KERNEL,
    "--descale" if DESCALE else "--no-descale",
    "--input-width", str(INPUT_WIDTH), "--input-height", str(INPUT_HEIGHT),
    "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD), "--pre-pad", str(PRE_PAD),
    "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
    "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
    "--tta", TTA, "--tta-batch-size", str(TTA_BATCH_SIZE),
    "--shift-ensemble", SHIFT_ENSEMBLE,
    "--residual-mode", RESIDUAL_MODE, "--residual-strength", str(RESIDUAL_STRENGTH),
    "--residual-flat-strength", str(RESIDUAL_FLAT_STRENGTH),
    "--residual-edge-strength", str(RESIDUAL_EDGE_STRENGTH),
    "--residual-edge-low", str(RESIDUAL_EDGE_LOW),
    "--residual-edge-high", str(RESIDUAL_EDGE_HIGH),
    "--base-correction", str(BASE_CORRECTION),
    "--back-projection-iterations", str(BACK_PROJECTION_ITERATIONS),
    "--back-projection-strength", str(BACK_PROJECTION_STRENGTH),
    "--back-projection-kernel", BACK_PROJECTION_KERNEL,
    "--back-projection-clamp", str(BACK_PROJECTION_CLAMP),
    "--dehalo-strength", str(DEHALO_STRENGTH), "--dehalo-radius", str(DEHALO_RADIUS),
    "--range-limit", str(RANGE_LIMIT), "--range-radius", str(RANGE_RADIUS),
    "--overshoot", str(OVERSHOOT), "--undershoot", str(UNDERSHOOT),
    "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
    "--anime4k" if ANIME4K else "--no-anime4k",
    "--anime4k-shader-dir", ANIME4K_SHADER_DIR,
    "--anime4k-shaders", ANIME4K_SHADERS, "--anime4k-strength", str(ANIME4K_STRENGTH),
    "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg", "--ffprobe-bin", "ffprobe",
    *EXTRA_ARGS,
]
print("[command]", shlex.join(command), flush=True)
subprocess.run(command, check=True)


## Optional Descale/getnative installation
Run this only when `DESCALE=True`. No ordinary resize fallback is used.


In [ ]:
import sys
if sys.version_info < (3, 12):
    raise RuntimeError("The pinned optional VapourSynth packages require Python 3.12 or newer.")
!pip install -q wrapt VapourSynth==77 vapoursynth-descale==12 vapoursynth-ffms2==5.2.1 getnative==3.3.0
!vapoursynth config
!vapoursynth check-env
!command -v vspipe
!getnative --help >/dev/null
